In [4]:
import pandas as pd
import os
import glob

# Folder paths
combined_base_folder = r"\\......\Tong_computer\Dissertation Study 2\EEG data\Combined Files\combined_power_entropy_video"
q2_y_folder = r"\\......\Tong_computer\Dissertation Study 2\EEG data\ttl_from_set\new_ttl\emotion_ttl"
q3_y_folder = r"\\......\Tong_computer\Dissertation Study 2\EEG data\ttl_from_set\new_ttl\empathy_ttl"
q4_y_folder = r"\\......\Tong_computer\Dissertation Study 2\EEG data\ttl_from_set\new_ttl\video_type_ttl"
output_folder = r"\\......\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files"

# Get list of combined_base_files
combined_base_files = glob.glob(os.path.join(combined_base_folder, "*.csv"))

# Function to process a subject's file
def process_subject_file(subject_num, combined_base_file):
    print(f"\nProcessing Subject {subject_num}...")

    # Identify corresponding q2_y, q3_y, and q4_y files
    q2_y_file = os.path.join(q2_y_folder, f"{subject_num}_modified_events_q2.txt")
    q3_y_file = os.path.join(q3_y_folder, f"{subject_num}_modified_events_q3.txt")
    q4_y_file = os.path.join(q4_y_folder, f"{subject_num}_modified_events_q4.txt")

    # Check if all necessary files exist
    if not all(os.path.exists(f) for f in [q2_y_file, q3_y_file, q4_y_file]):
        print(f"Skipping Subject {subject_num} - Missing q2_y, q3_y, or q4_y file.")
        return

    # Load the base file
    df = pd.read_csv(combined_base_file)

    # Identify chunks where "emotional" or "neutral" continues until "None" appears
    chunk_id = 0
    chunk_labels = []
    previous_value = None
    in_chunk = False  # Flag to indicate if we are inside a chunk

    for value in df["Channel 1 - Emotion_Condition"]:
        if value in ["emotional", "neutral"]:
            if not in_chunk:  # Start a new chunk when switching from "None"
                chunk_id += 1
                in_chunk = True
            chunk_labels.append(chunk_id)
        else:
            in_chunk = False  # End the current chunk when encountering "None"
            chunk_labels.append(None)

    df["chunk"] = chunk_labels  # Assign chunk labels to DataFrame

    # Count chunk sizes and maintain order
    chunk_sizes = df[df["chunk"].notna()].groupby("chunk").size().reset_index(name="size")

    # Print each chunk's size
    for i in range(len(chunk_sizes)):
        print(f"Chunk {int(chunk_sizes.loc[i, 'chunk'])}: {chunk_sizes.loc[i, 'size']} rows")

    # Function to process q files
    def process_q_file(q_file, column_name):
        # Load q file
        q_df = pd.read_csv(q_file, delim_whitespace=True)

        # Extract the 'type' column starting from the 3rd row (ignoring the header)
        q_types = q_df["type"].iloc[2:].reset_index(drop=True)

        # Ensure unique_chunks maintains the original order
        unique_chunks = df[df["chunk"].notna()].groupby("chunk", sort=False)["Channel 1 - Emotion_Condition"].first().reset_index()

        # Assign q values sequentially while preserving order
        for i in range(len(unique_chunks)):
            if i < len(q_types):  # Ensure we don't go out of bounds
                assigned_value = q_types.iloc[i]
                print(f"Processing Chunk {int(unique_chunks.loc[i, 'chunk'])}: "
                      f"Assigning value from row {i+3} in {column_name} -> {assigned_value}")
                unique_chunks.loc[i, column_name] = assigned_value

        return unique_chunks[["chunk", column_name]]

    # Process q2_y, q3_y, and q4_y
    q2_data = process_q_file(q2_y_file, "q2_y")
    q3_data = process_q_file(q3_y_file, "q3_y")
    q4_data = process_q_file(q4_y_file, "q4_y")

    # Merge q2_y, q3_y, and q4_y values back into the main DataFrame
    df = df.merge(q2_data, on="chunk", how="left")
    df = df.merge(q3_data, on="chunk", how="left")
    df = df.merge(q4_data, on="chunk", how="left")

    # Drop the chunk column as it's no longer needed
    df.drop(columns=["chunk"], inplace=True)

    # Save the updated file
    output_path = os.path.join(output_folder, f"{subject_num}_processed.csv")
    df.to_csv(output_path, index=False)

    print(f"File saved: {output_path}")

# Process all subject files
for combined_base_file in combined_base_files:
    # Extract subject number (first two digits of the filename)
    subject_num = os.path.basename(combined_base_file)[:2]

    # Process this subject
    process_subject_file(subject_num, combined_base_file)

print("\nProcessing complete for all subjects!")



Processing Subject 01...
Chunk 1: 17 rows
Chunk 2: 16 rows
Chunk 3: 14 rows
Chunk 4: 13 rows
Chunk 5: 13 rows
Chunk 6: 14 rows
Chunk 7: 14 rows
Chunk 8: 13 rows
Chunk 9: 16 rows
Chunk 10: 13 rows
Chunk 11: 17 rows
Chunk 12: 14 rows
Chunk 13: 19 rows
Chunk 14: 16 rows
Chunk 15: 16 rows
Chunk 16: 13 rows
Chunk 17: 14 rows
Chunk 18: 17 rows
Chunk 19: 13 rows
Chunk 20: 15 rows
Chunk 21: 13 rows
Chunk 22: 13 rows
Chunk 23: 15 rows
Chunk 24: 17 rows
Chunk 25: 14 rows
Chunk 26: 14 rows
Chunk 27: 14 rows
Chunk 28: 18 rows
Chunk 29: 8 rows
Chunk 30: 15 rows
Chunk 31: 16 rows
Chunk 32: 17 rows
Chunk 33: 20 rows
Chunk 34: 16 rows
Chunk 35: 15 rows
Chunk 36: 12 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> emotional
Processing Chunk 2: Assigning value from row 4 in q2_y -> neutral
Processing Chunk 3: Assigning value from row 5 in q2_y -> neutral
Processing Chunk 4: Assigning value from row 6 in q2_y -> emotional
Processing Chunk 5: Assigning value from row 7 in q2_y -> emotional


File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\02_processed.csv

Processing Subject 03...
Chunk 1: 19 rows
Chunk 2: 16 rows
Chunk 3: 14 rows
Chunk 4: 15 rows
Chunk 5: 16 rows
Chunk 6: 14 rows
Chunk 7: 13 rows
Chunk 8: 14 rows
Chunk 9: 13 rows
Chunk 10: 14 rows
Chunk 11: 14 rows
Chunk 12: 16 rows
Chunk 13: 20 rows
Chunk 14: 16 rows
Chunk 15: 15 rows
Chunk 16: 17 rows
Chunk 17: 14 rows
Chunk 18: 13 rows
Chunk 19: 18 rows
Chunk 20: 13 rows
Chunk 21: 8 rows
Chunk 22: 17 rows
Chunk 23: 16 rows
Chunk 24: 14 rows
Chunk 25: 17 rows
Chunk 26: 15 rows
Chunk 27: 14 rows
Chunk 28: 17 rows
Chunk 29: 14 rows
Chunk 30: 16 rows
Chunk 31: 13 rows
Chunk 32: 15 rows
Chunk 33: 17 rows
Chunk 34: 15 rows
Chunk 35: 13 rows
Chunk 36: 9 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> neutral
Processing Chunk 2: Assigning value from row 4 in q2_y -> emotional
Processing Chunk 3: Assigning value from row 5 in q

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\03_processed.csv

Processing Subject 04...
Chunk 1: 14 rows
Chunk 2: 16 rows
Chunk 3: 8 rows
Chunk 4: 14 rows
Chunk 5: 16 rows
Chunk 6: 13 rows
Chunk 7: 14 rows
Chunk 8: 14 rows
Chunk 9: 13 rows
Chunk 10: 17 rows
Chunk 11: 14 rows
Chunk 12: 16 rows
Chunk 13: 20 rows
Chunk 14: 15 rows
Chunk 15: 14 rows
Chunk 16: 13 rows
Chunk 17: 14 rows
Chunk 18: 19 rows
Chunk 19: 16 rows
Chunk 20: 14 rows
Chunk 21: 17 rows
Chunk 22: 15 rows
Chunk 23: 14 rows
Chunk 24: 18 rows
Chunk 25: 14 rows
Chunk 26: 16 rows
Chunk 27: 13 rows
Chunk 28: 16 rows
Chunk 29: 17 rows
Chunk 30: 17 rows
Chunk 31: 13 rows
Chunk 32: 17 rows
Chunk 33: 15 rows
Chunk 34: 13 rows
Chunk 35: 13 rows
Chunk 36: 13 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> emotional
Processing Chunk 2: Assigning value from row 4 in q2_y -> neutral
Processing Chunk 3: Assigning value from row 5 in 

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\04_processed.csv

Processing Subject 05...
Chunk 1: 14 rows
Chunk 2: 19 rows
Chunk 3: 13 rows
Chunk 4: 14 rows
Chunk 5: 15 rows
Chunk 6: 13 rows
Chunk 7: 16 rows
Chunk 8: 14 rows
Chunk 9: 13 rows
Chunk 10: 16 rows
Chunk 11: 16 rows
Chunk 12: 18 rows
Chunk 13: 14 rows
Chunk 14: 13 rows
Chunk 15: 17 rows
Chunk 16: 13 rows
Chunk 17: 13 rows
Chunk 18: 17 rows
Chunk 19: 17 rows
Chunk 20: 14 rows
Chunk 21: 13 rows
Chunk 22: 17 rows
Chunk 23: 16 rows
Chunk 24: 8 rows
Chunk 25: 16 rows
Chunk 26: 14 rows
Chunk 27: 14 rows
Chunk 28: 15 rows
Chunk 29: 16 rows
Chunk 30: 14 rows
Chunk 31: 15 rows
Chunk 32: 15 rows
Chunk 33: 13 rows
Chunk 34: 20 rows
Chunk 35: 17 rows
Chunk 36: 12 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> emotional
Processing Chunk 2: Assigning value from row 4 in q2_y -> neutral
Processing Chunk 3: Assigning value from row 5 in 

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\05_processed.csv

Processing Subject 06...
Chunk 1: 18 rows
Chunk 2: 16 rows
Chunk 3: 15 rows
Chunk 4: 14 rows
Chunk 5: 15 rows
Chunk 6: 16 rows
Chunk 7: 14 rows
Chunk 8: 16 rows
Chunk 9: 15 rows
Chunk 10: 8 rows
Chunk 11: 17 rows
Chunk 12: 13 rows
Chunk 13: 17 rows
Chunk 14: 17 rows
Chunk 15: 14 rows
Chunk 16: 13 rows
Chunk 17: 16 rows
Chunk 18: 13 rows
Chunk 19: 14 rows
Chunk 20: 14 rows
Chunk 21: 14 rows
Chunk 22: 17 rows
Chunk 23: 14 rows
Chunk 24: 19 rows
Chunk 25: 15 rows
Chunk 26: 13 rows
Chunk 27: 20 rows
Chunk 28: 14 rows
Chunk 29: 14 rows
Chunk 30: 16 rows
Chunk 31: 16 rows
Chunk 32: 13 rows
Chunk 33: 17 rows
Chunk 34: 15 rows
Chunk 35: 13 rows
Chunk 36: 9 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> neutral
Processing Chunk 2: Assigning value from row 4 in q2_y -> emotional
Processing Chunk 3: Assigning value from row 5 in q

...

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\66_processed.csv

Processing Subject 67...
Chunk 1: 13 rows
Chunk 2: 16 rows
Chunk 3: 16 rows
Chunk 4: 16 rows
Chunk 5: 17 rows
Chunk 6: 17 rows
Chunk 7: 14 rows
Chunk 8: 13 rows
Chunk 9: 14 rows
Chunk 10: 15 rows
Chunk 11: 14 rows
Chunk 12: 14 rows
Chunk 13: 14 rows
Chunk 14: 17 rows
Chunk 15: 18 rows
Chunk 16: 13 rows
Chunk 17: 13 rows
Chunk 18: 19 rows
Chunk 19: 8 rows
Chunk 20: 17 rows
Chunk 21: 17 rows
Chunk 22: 14 rows
Chunk 23: 14 rows
Chunk 24: 14 rows
Chunk 25: 13 rows
Chunk 26: 15 rows
Chunk 27: 14 rows
Chunk 28: 15 rows
Chunk 29: 16 rows
Chunk 30: 16 rows
Chunk 31: 16 rows
Chunk 32: 14 rows
Chunk 33: 13 rows
Chunk 34: 15 rows
Chunk 35: 13 rows
Chunk 36: 17 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> emotional
Processing Chunk 2: Assigning value from row 4 in q2_y -> neutral
Processing Chunk 3: Assigning value from row 5 in 

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\67_processed.csv

Processing Subject 68...
Chunk 1: 14 rows
Chunk 2: 14 rows
Chunk 3: 14 rows
Chunk 4: 19 rows
Chunk 5: 16 rows
Chunk 6: 15 rows
Chunk 7: 8 rows
Chunk 8: 13 rows
Chunk 9: 17 rows
Chunk 10: 16 rows
Chunk 11: 14 rows
Chunk 12: 15 rows
Chunk 13: 16 rows
Chunk 14: 14 rows
Chunk 15: 16 rows
Chunk 16: 13 rows
Chunk 17: 14 rows
Chunk 18: 17 rows
Chunk 19: 13 rows
Chunk 20: 17 rows
Chunk 21: 20 rows
Chunk 22: 16 rows
Chunk 23: 18 rows
Chunk 24: 17 rows
Chunk 25: 13 rows
Chunk 26: 14 rows
Chunk 27: 14 rows
Chunk 28: 15 rows
Chunk 29: 14 rows
Chunk 30: 14 rows
Chunk 31: 17 rows
Chunk 32: 16 rows
Chunk 33: 15 rows
Chunk 34: 13 rows
Chunk 35: 13 rows
Chunk 36: 10 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> emotional
Processing Chunk 2: Assigning value from row 4 in q2_y -> neutral
Processing Chunk 3: Assigning value from row 5 in 

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\68_processed.csv

Processing Subject 69...
Chunk 1: 17 rows
Chunk 2: 14 rows
Chunk 3: 16 rows
Chunk 4: 14 rows
Chunk 5: 15 rows
Chunk 6: 13 rows
Chunk 7: 19 rows
Chunk 8: 17 rows
Chunk 9: 14 rows
Chunk 10: 13 rows
Chunk 11: 13 rows
Chunk 12: 16 rows
Chunk 13: 14 rows
Chunk 14: 13 rows
Chunk 15: 14 rows
Chunk 16: 16 rows
Chunk 17: 13 rows
Chunk 18: 15 rows
Chunk 19: 16 rows
Chunk 20: 16 rows
Chunk 21: 14 rows
Chunk 22: 14 rows
Chunk 23: 29 rows
Chunk 24: 17 rows
Chunk 25: 17 rows
Chunk 26: 14 rows
Chunk 27: 17 rows
Chunk 28: 20 rows
Chunk 29: 18 rows
Chunk 30: 15 rows
Chunk 31: 15 rows
Chunk 32: 8 rows
Chunk 33: 13 rows
Chunk 34: 15 rows
Chunk 35: 9 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> emotional
Processing Chunk 2: Assigning value from row 4 in q2_y -> neutral
Processing Chunk 3: Assigning value from row 5 in q2_y -> emotional
P

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\70_processed.csv

Processing Subject 71...
Chunk 1: 17 rows
Chunk 2: 13 rows
Chunk 3: 13 rows
Chunk 4: 14 rows
Chunk 5: 15 rows
Chunk 6: 17 rows
Chunk 7: 16 rows
Chunk 8: 13 rows
Chunk 9: 14 rows
Chunk 10: 14 rows
Chunk 11: 8 rows
Chunk 12: 20 rows
Chunk 13: 14 rows
Chunk 14: 17 rows
Chunk 15: 16 rows
Chunk 16: 16 rows
Chunk 17: 15 rows
Chunk 18: 13 rows
Chunk 19: 16 rows
Chunk 20: 16 rows
Chunk 21: 15 rows
Chunk 22: 14 rows
Chunk 23: 14 rows
Chunk 24: 14 rows
Chunk 25: 13 rows
Chunk 26: 13 rows
Chunk 27: 14 rows
Chunk 28: 15 rows
Chunk 29: 17 rows
Chunk 30: 19 rows
Chunk 31: 13 rows
Chunk 32: 16 rows
Chunk 33: 18 rows
Chunk 34: 15 rows
Chunk 35: 14 rows
Chunk 36: 13 rows
Processing Chunk 1: Assigning value from row 3 in q2_y -> emotional
Processing Chunk 2: Assigning value from row 4 in q2_y -> emotional
Processing Chunk 3: Assigning value from row 5 i

File saved: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\71_processed.csv

Processing complete for all subjects!


In [3]:
################## Changing column names, adding time stamps and filling up empty cells ##############
import pandas as pd
import os
import glob

# Folder paths
input_folder = r"\\......\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\processed"
output_folder = r"\\......\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals"

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Define columns to drop
columns_to_delete = [
    "Channel 1 - Event_Number", "Quality", "Correct Time", "Chunk Index", "stimID", "Time (s)"
]

# Define channel mapping
channel_mapping = {
    "Channel 1": "AF3",
    "Channel 2": "F7",
    "Channel 3": "F3",
    "Channel 4": "FC5",
    "Channel 5": "T7",
    "Channel 6": "P7",
    "Channel 7": "O1",
    "Channel 8": "O2",
    "Channel 9": "P8",
    "Channel 10": "T8",
    "Channel 11": "FC6",
    "Channel 12": "F4",
    "Channel 13": "F8",
    "Channel 14": "AF4",
    "Channel 1 - Emotion_Condition": "q1_y"  # Rename q1_y
}

# Get list of all CSV files in the input folder
input_files = glob.glob(os.path.join(input_folder, "*.csv"))

# Process each file
for file_path in input_files:
    file_name = os.path.basename(file_path)
    new_file_name = file_name.replace(".csv", "_final.csv")  # Append "_final" before ".csv"

    print(f"Processing {file_name}...")

    # Load CSV
    df = pd.read_csv(file_path)

    # Remove rows where "Channel 1 - Emotion_Condition" is "None"
    df = df[df["Channel 1 - Emotion_Condition"] != "None"]

    # Drop unwanted columns if they exist
    df = df.drop(columns=[col for col in columns_to_delete if col in df.columns], errors='ignore')

    # Rename columns
    df = df.rename(columns=channel_mapping)

    # Ensure q2_y, q3_y, q4_y exist before assignment
    for col in ["q2_y", "q3_y", "q4_y"]:
        if col not in df.columns:
            df[col] = ""

    # Copy `long_happy`, `long_sad`, or `fixation` values to `q2_y`, `q3_y`, and `q4_y`
    mask = df["q1_y"].isin(["long_happy", "long_sad", "fixation"])
    #df.loc[mask, ["q2_y", "q3_y", "q4_y"]] = df.loc[mask, "q1_y"].apply(lambda x: [x])
    # Copy values from q1_y to q2_y, q3_y, and q4_y individually
    df.loc[mask, "q2_y"] = df.loc[mask, "q1_y"]
    df.loc[mask, "q3_y"] = df.loc[mask, "q1_y"]
    df.loc[mask, "q4_y"] = df.loc[mask, "q1_y"]


    # Create condition time stamps column
    condition_timestamps = []
    counter = 0
    previous_q1_y = None

    for i, row in df.iterrows():
        if row["q1_y"] == "long_happy":  # Reset counter at these conditions
            counter += 1
        elif row["q1_y"] == "long_sad":  # Reset counter at these conditions
            if counter == 60 and previous_q1_y == "long_happy":
                counter=1
            else:
                counter += 1
        elif row["q1_y"] == "fixation":  # Fixation resets to 0
            counter = 0
        else:  # Increment during "emotional" or "neutral"
            if counter == 0:
                counter = 1  # Restart from 1 if previously at 0
            else:
                counter += 1
        
        condition_timestamps.append(counter)
        # Update previous_q1_y for the next iteration
        previous_q1_y = row["q1_y"]

    df["condition time stamps"] = condition_timestamps  # Add to dataframe

    # Save the cleaned file with "_final" in the name
    output_path = os.path.join(output_folder, new_file_name)
    df.to_csv(output_path, index=False)
    
    print(f"Saved cleaned file: {output_path}")

print("\nProcessing complete for all files!")


Processing 01_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\01_processed_final.csv
Processing 02_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\02_processed_final.csv
Processing 03_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\03_processed_final.csv
Processing 04_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\04_processed_final.csv
Processing 05_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Fil

Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\40_processed_final.csv
Processing 41_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\41_processed_final.csv
Processing 42_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\42_processed_final.csv
Processing 43_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual files\final individuals\43_processed_final.csv
Processing 44_processed.csv...
Saved cleaned file: \\files.brandeis.edu\gutsell-lab\Tong_computer\Dissertation Study 2\EEG data\Combined Files\final cleaned individual fil